In [0]:
%sh
nc -zv c93366692d52454d9e959ac3f2dc9cb6.eastus.azure.elastic-cloud.com 443

Connection to c93366692d52454d9e959ac3f2dc9cb6.eastus.azure.elastic-cloud.com (52.191.218.117) 443 port [tcp/https] succeeded!


In [0]:
%sh curl -s ifconfig.me

52.249.199.78

In [0]:
from sdds.common.util import NotebookUtil
from pyspark.sql.functions import col, from_json, explode_outer, lit
from pyspark.sql.types import ArrayType, StringType
from datetime import datetime

# Parameter widget for extraction date partition
dbutils.widgets.text("extraction_date", datetime.now().strftime("%Y-%m-%d"), "Extraction Date")
extraction_date = dbutils.widgets.get("extraction_date")

catalog_name = NotebookUtil.notebook_param("sdds_catalog")
bronze_schema_name = NotebookUtil.notebook_param("sdds_bronze_schema")
silver_schema_name = NotebookUtil.notebook_param("sdds_silver_schema")
bronze_table_name = "catalog-load-inventory-dbx-bronze"
silver_table_name = "catalog-load-sku"

In [0]:
from pyspark.sql.functions import col, from_json, explode_outer, to_timestamp, to_date, row_number, concat_ws, substring, lit
from pyspark.sql.window import Window
from pyspark.sql.types import (
    ArrayType, StringType, StructType, StructField, IntegerType
)

# --- Schema definitions for nested JSON fields ---
dsgInventory_schema = ArrayType(StructType([
    StructField("location", IntegerType(), True),
    StructField("isaqty", IntegerType(), True),
    StructField("atsqty", IntegerType(), True),
    StructField("boplqty", IntegerType(), True),
    StructField("time", StringType(), True)
]))

defSkuParent_schema = StructType([
    StructField("partnumber", StringType(), True),
    StructField("parentPartnumber", StringType(), True),
    StructField("skuInvParentPartNumber", StringType(), True),
    StructField("skuInvPartNumber", StringType(), True),
])

# --- Read bronze (filtered by extraction_date partition) and deduplicate by partnumber ---
bronze_raw_df = (
    spark.read.table(f"`{catalog_name}`.`{bronze_schema_name}`.`{bronze_table_name}`")
    .filter(col("extraction_date") == extraction_date)
)
_w = Window.partitionBy("partnumber").orderBy(col("load_timestamp").desc())
bronze_df = (
    bronze_raw_df
    .withColumn("_rn", row_number().over(_w))
    .filter(col("_rn") == 1)
    .drop("_rn")
)
print(f"Reading bronze partition for extraction_date = {extraction_date}")
print(f"Bronze records (raw): {bronze_raw_df.count():,}  |  after dedup: {bronze_df.count():,}")

# --- Helper: reformat 'yyyyMMddHHmmssSSS' (e.g. '20260725061651127') into
# a standard 'yyyy-MM-dd HH:mm:ss.SSS' string. Spark 3's strict new-parser
# rejects this compact numeric layout via to_timestamp(..., "yyyyMMddHHmmssSSS")
# (SparkUpgradeException / INCONSISTENT_BEHAVIOR_CROSS_VERSION), so we build
# an ISO-formatted string ourselves and let to_timestamp parse that instead.
def parse_compact_time(raw_col):
    return to_timestamp(
        concat_ws(
            "",
            substring(raw_col, 1, 4), lit("-"),
            substring(raw_col, 5, 2), lit("-"),
            substring(raw_col, 7, 2), lit(" "),
            substring(raw_col, 9, 2), lit(":"),
            substring(raw_col, 11, 2), lit(":"),
            substring(raw_col, 13, 2), lit("."),
            substring(raw_col, 15, 3)
        )
    )

# --- Step 1: Parse nested JSON into typed columns ---
parsed_df = (
    bronze_df
    .select(
        col("partnumber"),
        col("parentPartnumber"),
        col("skuInvParentPartNumber"),
        col("skuInvPartNumber"),
        from_json(col("inventory"), dsgInventory_schema).alias("inventory"),
        col("load_timestamp")
    )
)

# --- Step 2: Main silver table (sku/parent linkage, NO arrays) ---
# This is what lets you join inventory back to other silver tables on partnumber/parentPartnumber
silver_df = (
    parsed_df
    .select(
        "partnumber",
        "parentPartnumber",
        "skuInvParentPartNumber",
        "skuInvPartNumber",
        "load_timestamp",
        to_date(col("load_timestamp")).alias("extraction_date")
    )
)

# --- Step 3: Exploded table (one row per inventory element, flat columns) ---
# inventory: Array<Struct> → partnumber | parentPartnumber | location | isaqty | atsqty | boplqty | time
silver_inventory_df = (
    parsed_df.select(
        "partnumber", "parentPartnumber", "load_timestamp",
        explode_outer(col("inventory")).alias("elem")
    )
    .select(
        "partnumber",
        "parentPartnumber",
        col("elem.location"),
        col("elem.isaqty"),
        col("elem.atsqty"),
        col("elem.boplqty"),
        parse_compact_time(col("elem.time")).alias("time"),
        to_date(col("load_timestamp")).alias("extraction_date"),
        col("load_timestamp")
    )
)

print("=== Silver DataFrames created ===")
print(f"  silver_df (main):        {silver_df.columns}")
print(f"  silver_inventory_df:     {silver_inventory_df.columns}")


Reading bronze partition for extraction_date = 2026-07-28
Bronze records (raw): 685,283  |  after dedup: 685,283
=== Silver DataFrames created ===
  silver_df (main):        ['partnumber', 'parentPartnumber', 'skuInvParentPartNumber', 'skuInvPartNumber', 'load_timestamp', 'extraction_date']
  silver_inventory_df:     ['partnumber', 'parentPartnumber', 'location', 'isaqty', 'atsqty', 'boplqty', 'time', 'extraction_date', 'load_timestamp']


In [0]:
silver_df.display()


partnumber,parentPartnumber,skuInvParentPartNumber,skuInvPartNumber,load_timestamp,extraction_date
10075043,17THOULGHTHKRXXXXAPA,17THOULGHTHKRXXXXAPA,10075043,2026-07-28T20:18:57Z,2026-07-28
10142519,15SRSMNFLCCMBSCRFAPA,15SRSMNFLCCMBSCRFAPA,10142519,2026-07-28T20:18:57Z,2026-07-28
10155317,23CTTMK87SSPCKTTXMOA,23CTTMK87SSPCKTTXMOA,10155317,2026-07-28T20:18:57Z,2026-07-28
10155676,15CTTMCRYLCWTCHCPAPA,15CTTMCRYLCWTCHCPAPA,10155676,2026-07-28T20:18:57Z,2026-07-28
10155677,15CTTMCRYLCWTCHCPAPA,15CTTMCRYLCWTCHCPAPA,10155677,2026-07-28T20:18:57Z,2026-07-28
10320120,16MRRMJNGLMCBLCKXFOT,16MRRMJNGLMCBLCKXFOT,10320120,2026-07-28T20:18:57Z,2026-07-28
10332563,16WILUJTVLTNXXXXXBKB,16WILUJTVLTNXXXXXBKB,10332563,2026-07-28T20:18:57Z,2026-07-28
10394242,16LOGUNCGRGSTDMSTHDG,16LOGUNCGRGSTDMSTHDG,10394242,2026-07-28T20:18:57Z,2026-07-28
10394250,16LOGUNCMZZSTDMSTHDG,16LOGUNCMZZSTDMSTHDG,10394250,2026-07-28T20:18:57Z,2026-07-28
10394255,16LOGUNCSCRSTDMSTHDG,16LOGUNCSCRSTDMSTHDG,10394255,2026-07-28T20:18:57Z,2026-07-28


In [0]:
silver_inventory_df.display()


partnumber,parentPartnumber,location,isaqty,atsqty,boplqty,time,extraction_date,load_timestamp
10075043,17THOULGHTHKRXXXXAPA,0,0,392,0,2026-07-25T15:27:43.389Z,2026-07-28,2026-07-28T20:18:57Z
10075043,17THOULGHTHKRXXXXAPA,-1,0,0,0,2026-07-23T21:30:49.326Z,2026-07-28,2026-07-28T20:18:57Z
10075043,17THOULGHTHKRXXXXAPA,5202,0,0,0,2026-07-23T21:30:48.784Z,2026-07-28,2026-07-28T20:18:57Z
10142519,15SRSMNFLCCMBSCRFAPA,1341,3,3,0,2026-04-22T04:56:07.552Z,2026-07-28,2026-07-28T20:18:57Z
10142519,15SRSMNFLCCMBSCRFAPA,1583,2,2,0,2026-04-22T05:04:48.318Z,2026-07-28,2026-07-28T20:18:57Z
10142519,15SRSMNFLCCMBSCRFAPA,1582,2,2,0,2026-05-18T00:57:44.362Z,2026-07-28,2026-07-28T20:18:57Z
10142519,15SRSMNFLCCMBSCRFAPA,1581,4,4,0,2026-05-31T20:15:55.307Z,2026-07-28,2026-07-28T20:18:57Z
10142519,15SRSMNFLCCMBSCRFAPA,1580,2,2,0,2026-04-22T05:04:48.317Z,2026-07-28,2026-07-28T20:18:57Z
10142519,15SRSMNFLCCMBSCRFAPA,1217,1,1,0,2026-07-16T18:31:12.133Z,2026-07-28,2026-07-28T20:18:57Z
10142519,15SRSMNFLCCMBSCRFAPA,1337,1,1,0,2026-07-10T20:17:56.652Z,2026-07-28,2026-07-28T20:18:57Z


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{silver_schema_name}`")

# --- Write main silver table + standalone normalized inventory table ---
# Unlike the catalog-load-silver pattern, the inventory table is NOT joined back
# to silver_df — it's written standalone and can be joined downstream on
# partnumber / parentPartnumber whenever needed.

silver_tables = {
    "": silver_df,
    "-inventory": silver_inventory_df,
}

for suffix, df in silver_tables.items():
    table_full = f"`{catalog_name}`.`{silver_schema_name}`.`{silver_table_name}{suffix}`"
    table_exists = spark.catalog.tableExists(f"{catalog_name}.{silver_schema_name}.`{silver_table_name}{suffix}`")

    if not table_exists:
        # First run: create table with extraction_date partition
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("extraction_date")
            .saveAsTable(table_full)
        )
    else:
        # Subsequent runs: overwrite only this partition (idempotent re-runs)
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"extraction_date = '{extraction_date}'")
            .option("mergeSchema", "true")
            .partitionBy("extraction_date")
            .saveAsTable(table_full)
        )
    print(f"  Written {df.count():,} rows to {silver_table_name}{suffix}")

print("\nAll silver tables written successfully.")


  Written 685,283 rows to catalog-load-sku
  Written 69,075,322 rows to catalog-load-sku-inventory

All silver tables written successfully.
